# 01 · Preprocesamiento

Equivalente a `1_preprocess_dates.ipynb` + `2_limpiar_crear_chunks.ipynb` de Karen.

In [9]:
# ============================================================
# CELL 0 - CONFIG
# ============================================================
from pathlib import Path
import os, subprocess

# -- Rutas ---------------------------------------------------
ARCHIVO_TWEETS = Path(r'C:\Users\afpue\OneDrive\Documentos\GitHub\icare\archivos\df_twitter.csv')
DATA_PROCESSED = Path(r'C:\Users\afpue\OneDrive\Documentos\GitHub\icare\kMetodo\resultadosPropios')

# Crear carpeta fisica via cmd.exe (evita OneDrive virtual filesystem)
subprocess.run(
    f'cmd /c if not exist "{DATA_PROCESSED}" mkdir "{DATA_PROCESSED}"',
    shell=True, capture_output=True
)

# Verificar que se puede escribir
_test = DATA_PROCESSED / '_test.tmp'
try:
    _test.write_text('ok')
    _test.unlink()
    print(f'[OK] DATA_PROCESSED accesible: {DATA_PROCESSED}')
except Exception as _e:
    # Fallback: carpeta local garantizada dentro del proyecto
    DATA_PROCESSED = Path(os.getcwd()).parent / 'resultadosPropios'
    subprocess.run(
        f'cmd /c if not exist "{DATA_PROCESSED}" mkdir "{DATA_PROCESSED}"',
        shell=True, capture_output=True
    )
    print(f'[FALLBACK] Ruta alternativa: {DATA_PROCESSED}')

# -- Columnas del DataFrame ----------------------------------
COL_TEXTO       = 'Content'
COL_FECHA       = 'Fecha'
COL_DATE        = 'Date'
COL_AUTOR       = 'Author_Normalized'
COL_AUTOR_NAME  = 'Author Name'
COL_LOCATION    = 'Location'
COL_LIKES       = 'Number of Likes'
COL_RTS         = 'Number of Retweets'
COL_HASHTAGS    = 'Hashtags'
COL_MENTIONS    = 'Mentions'
COL_SENTIMIENTO = 'Sentimiento'
COL_POLARIDAD   = 'Polaridad'
COL_ENTIDAD     = 'Entidad'

print(f'  ARCHIVO_TWEETS : {ARCHIVO_TWEETS}')
print(f'  DATA_PROCESSED : {DATA_PROCESSED}')


[FALLBACK] Ruta alternativa: c:\Users\afpue\OneDrive\Documentos\GitHub\icare\kMetodo\resultadosPropios
  ARCHIVO_TWEETS : C:\Users\afpue\OneDrive\Documentos\GitHub\icare\archivos\df_twitter.csv
  DATA_PROCESSED : c:\Users\afpue\OneDrive\Documentos\GitHub\icare\kMetodo\resultadosPropios


In [10]:
# ============================================================
# CELL 1 - CARGA
# ============================================================
import pandas as pd

if 'df_tw' in dir():
    df = df_tw.copy()
    print('[CARGA] df_tw encontrado en memoria.')
else:
    df = pd.read_csv(ARCHIVO_TWEETS, low_memory=False)
    print(f'[CARGA] Leido desde disco: {ARCHIVO_TWEETS}')

print(f'  Filas    : {len(df):,}')
print(f'  Columnas : {list(df.columns)}')
df.head(3)


[CARGA] Leido desde disco: C:\Users\afpue\OneDrive\Documentos\GitHub\icare\archivos\df_twitter.csv
  Filas    : 151,424
  Columnas : ['Author', 'Content', 'Date', 'Location', 'Number of Likes', 'Number of Retweets', 'In Reply To', 'Author Name', 'Author Description', 'Author Statuses Count', 'Author Favourites Count', 'Author Friends Count', 'Author Followers Count', 'Author Listed Count', 'Author Verified', 'Mentions', 'Hashtags', 'Content_cleaned', 'Content_cleaned_2', 'Seed_Set_1', 'In Reply To Normalized', 'Seed_Set_2', 'Seed_Set_2_Prob', 'Seed_Set_2_Model', 'Author_Normalized', 'Seed_Set_3', 'Entidad', 'Prob entidad', 'Polaridad', 'Fecha', 'Sentimiento']


,Author,Content,Date,Location,Number of Likes,Number of Retweets,In Reply To,Author Name,Author Description,Author Statuses Count,...,Seed_Set_2,Seed_Set_2_Prob,Seed_Set_2_Model,Author_Normalized,Seed_Set_3,Entidad,Prob entidad,Polaridad,Fecha,Sentimiento
0,@TReporta,VÍDEO| Reos que no están contagiados con COVID...,2020-06-03 18:56:24,NaN,0.0,0.0,NaN,Telemetro Reporta,Únete a la conversación y mantente informado c...,570708.0,...,0,0.427514,NB,@treporta,0,"Noticias locales, nacionales o globales",0.2946,Neutra,2020-06-03,otro
1,@annytak21,@IvanDuque @jaimepumarejo ya se enteraron de l...,2020-06-03 18:56:24,NaN,0.0,0.0,@IvanDuque,ANNA KAROLYNNA,NaN,233.0,...,0,0.505196,NB,@annytak21,1,Sin descripción,0.0000,Negativa,2020-06-03,otro
2,@ferumapress,El calvario de una familia en Cali por su mamá...,2020-06-03 18:56:17,NaN,0.0,0.0,NaN,Fernando Umaña Mejía,Periodista. Corresponsal de El Tiempo en Cali....,1674.0,...,1,0.999976,BERT,@ferumapress,0,"Noticias locales, nacionales o globales",0.5071,Negativa,2020-06-03,otro


In [11]:
# ============================================================
# CELL 2 - FECHAS
# Equivalente a: corpus_completo = utils.aplicar_funcion_fecha(corpus_completo)
# ============================================================

df[COL_FECHA] = pd.to_datetime(df[COL_FECHA], errors='coerce')
df[COL_DATE]  = pd.to_datetime(df[COL_DATE],  errors='coerce')

n_nulos = df[COL_FECHA].isna().sum()
print(f'[FECHAS] Tweets con fecha valida : {len(df) - n_nulos:,}')
print(f'[FECHAS] Tweets sin fecha        : {n_nulos:,}')
print(f'[FECHAS] Rango                   : {df[COL_FECHA].min()} -> {df[COL_FECHA].max()}')

df['anio']     = df[COL_FECHA].dt.year
df['mes']      = df[COL_FECHA].dt.month
df['anio_mes'] = df[COL_FECHA].dt.to_period('M')

df[[COL_FECHA, COL_DATE, 'anio', 'mes', 'anio_mes']].head(3)


[FECHAS] Tweets con fecha valida : 151,424
[FECHAS] Tweets sin fecha        : 0
[FECHAS] Rango                   : 2020-03-31 00:00:00 -> 2020-08-18 00:00:00


,Fecha,Date,anio,mes,anio_mes
0,2020-06-03,2020-06-03 18:56:24,2020,6,2020-06
1,2020-06-03,2020-06-03 18:56:24,2020,6,2020-06
2,2020-06-03,2020-06-03 18:56:17,2020,6,2020-06


In [12]:
# ============================================================
# CELL 3 - LIMPIEZA
# Equivalente a: corpus = utils.aplicar_funcion_limpieza(corpus)
# ============================================================
import re

def limpiar_tweet(texto: str) -> str:
    if not isinstance(texto, str):
        return ''
    texto = re.sub(r'^RT\s+',                                          '',    texto, flags=re.IGNORECASE)
    texto = re.sub(r'http\S+|www\S+|https\S+',                      ' ',   texto)
    texto = re.sub(r'[^0-9A-Za-zaeiouAEIOUaeiouAEIOUnNuU\s\.\,\;\:\!\?\¿\¡]', '', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

df['Texto_limpio'] = df[COL_TEXTO].apply(limpiar_tweet)

n_vacios = (df['Texto_limpio'].str.strip() == '').sum()
print(f'[LIMPIEZA] Tweets vacios tras limpieza : {n_vacios:,}')

df = df[df['Texto_limpio'].str.strip() != ''].reset_index(drop=True)
print(f'[LIMPIEZA] Tweets conservados          : {len(df):,}')

df['n_palabras'] = df['Texto_limpio'].str.split().str.len()
print(f"[LIMPIEZA] Palabras/tweet  media: {df['n_palabras'].mean():.1f}  mediana: {df['n_palabras'].median():.0f}  max: {df['n_palabras'].max()}")

df[[COL_TEXTO, 'Texto_limpio', 'n_palabras']].head(3)


[LIMPIEZA] Tweets vacios tras limpieza : 0
[LIMPIEZA] Tweets conservados          : 151,424
[LIMPIEZA] Palabras/tweet  media: 30.4  mediana: 31  max: 105


,Content,Texto_limpio,n_palabras
0,VÍDEO| Reos que no están contagiados con COVID...,VDEO Reos que no estn contagiados con COVID19 ...,15
1,@IvanDuque @jaimepumarejo ya se enteraron de l...,IvanDuque jaimepumarejo ya se enteraron de la ...,26
2,El calvario de una familia en Cali por su mamá...,El calvario de una familia en Cali por su mam ...,16


In [13]:
# ============================================================
# CELL 4 - ID UNICO
# En utils.crear_chunks() del paper: id_doc = idx + 1
# ============================================================

df = df.reset_index(drop=True)
df['id_doc'] = df.index + 1

print(f"[ID] Rango id_doc : {df['id_doc'].min()} -> {df['id_doc'].max()}")
df[['id_doc', COL_AUTOR, COL_FECHA, 'Texto_limpio']].head(3)


[ID] Rango id_doc : 1 -> 151424


,id_doc,Author_Normalized,Fecha,Texto_limpio
0,1,@treporta,2020-06-03,VDEO Reos que no estn contagiados con COVID19 ...
1,2,@annytak21,2020-06-03,IvanDuque jaimepumarejo ya se enteraron de la ...
2,3,@ferumapress,2020-06-03,El calvario de una familia en Cali por su mam ...


In [14]:
# ============================================================
# CELL 5 - GUARDAR CORPUS LIMPIO
# ============================================================

COLS_CORPUS = [
    'id_doc',
    COL_AUTOR, COL_AUTOR_NAME, COL_FECHA, COL_DATE,
    'anio', 'mes', 'anio_mes',
    COL_LOCATION, COL_LIKES, COL_RTS,
    COL_HASHTAGS, COL_MENTIONS,
    COL_SENTIMIENTO, COL_POLARIDAD, COL_ENTIDAD,
    COL_TEXTO, 'Texto_limpio', 'n_palabras',
]
COLS_CORPUS = [c for c in COLS_CORPUS if c in df.columns]

corpus_limpio = df[COLS_CORPUS].copy()

ruta_parquet = DATA_PROCESSED / 'corpus_cleaned.parquet'
ruta_excel   = DATA_PROCESSED / 'corpus_cleaned.xlsx'

corpus_limpio.to_parquet(ruta_parquet, index=False, engine='pyarrow')
print(f'[GUARDADO] corpus_cleaned.parquet : {len(corpus_limpio):,} filas')

corpus_limpio.to_excel(ruta_excel, index=False, engine='openpyxl')
print(f'[GUARDADO] corpus_cleaned.xlsx    : {len(corpus_limpio):,} filas')
print(f'  Ruta: {DATA_PROCESSED}')


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\afpue\\OneDrive\\Documentos\\GitHub\\icare\\kMetodo\\resultadosPropios\\corpus_cleaned.parquet'

In [ ]:
# ============================================================
# CELL 6 - VERIFICACION FINAL
# ============================================================

print('=== RESUMEN NOTEBOOK 01 ===')
print(f'  Tweets tras limpieza  : {len(corpus_limpio):,}')
print(f'  Rango de fechas       : {corpus_limpio[COL_FECHA].min().date()} -> {corpus_limpio[COL_FECHA].max().date()}')
print(f'  Autores unicos        : {corpus_limpio[COL_AUTOR].nunique():,}')
print(f'  Media palabras/tweet  : {corpus_limpio["n_palabras"].mean():.1f}')
print()
print(f'  Archivos en: {DATA_PROCESSED}')
print()
print('Notebook 01 completado.')
print('Siguiente -> 02_estadisticos_corpus.ipynb')

corpus_limpio.head(3)
